# Modelagem preditiva — risco de defasagem

**Alvo binário:** estudante em defasagem quando `defasagem < 0` (fase efetiva abaixo da ideal na base harmonizada).

**Pipeline reproduzível:** mesma lógica de `src/pede_model.py` (usada pelo Streamlit e por `scripts/train_model.py`).

1. [Base de dados](#0-base-de-dados) — parquet unificado  
2. [Engenharia de atributos](#1-engenharia-de-atributos-feature-engineering) — matriz `X`, vetor `y`, pré-processamento no `sklearn.Pipeline`  
3. [Separação treino / teste](#2-separação-em-treino-e-teste) — holdout estratificado  
4. [Modelagem preditiva](#3-modelagem-preditiva) — `RandomForestClassifier`  
5. [Avaliação](#4-avaliação-dos-resultados) — ROC-AUC, relatório de classificação, matriz de confusão  
6. [(Opcional) Exportar o modelo](#5-opcional-exportar-o-mesmo-artefato-do-app) — `risk_defasagem.joblib`

Execute a partir da **raiz do repositório** (a pasta que contém `src/` e `notebooks/`). A primeira célula de código ajusta `sys.path`.

## 0. Base de dados

Carregamos `data_processed/pede_unificado.parquet`. Se ainda não existir, `ensure_parquet` gera a partir dos CSVs via `src/pede_cleaning.build_unified` (mesmo fluxo do app).

In [ ]:
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.pede_model import ensure_parquet

pq = ensure_parquet(ROOT)
df = pd.read_parquet(pq)
print("Parquet:", pq)
print("Shape:", df.shape)
df[["ra", "ano_cohorte", "defasagem"]].head()

## 1. Engenharia de atributos (feature engineering)

- **Definição do alvo:** `y = 1` se `defasagem < 0`, caso contrário `0` (função `build_xy`).
- **Entradas (`X`):** listas `NUMERIC_FEATURES` e `CATEGORICAL_FEATURES` — **não** usamos `ian` nem `defasagem` como features (evita vazamento do alvo).
- **Categórica:** `genero` como string; vazios viram `"Desconhecido"`.
- **Filtro de linhas:** mantemos apenas linhas com **pelo menos um** indicador numérico não nulo (igual a `train_risk_model` em `pede_model.py`).
- **Pré-processamento no modelo:** dentro do `Pipeline` — imputação mediana + `StandardScaler` nos numéricos; imputação moda + `OneHotEncoder` em `genero`.

In [ ]:
import numpy as np

from src.pede_model import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_xy,
)

X, y = build_xy(df)
mask = X[NUMERIC_FEATURES].notna().any(axis=1)
X = X.loc[mask]
y = y[mask.values]

print("Features numéricas (", len(NUMERIC_FEATURES), "):", NUMERIC_FEATURES, sep="\n")
print("\nFeatures categóricas:", CATEGORICAL_FEATURES)
print("\nTaxa de positivos (defasagem < 0):", float(y.mean()))
X.head()

## 2. Separação em treino e teste

Holdout **25%** para teste, `random_state=42`, **estratificado** por `y` — mesmos hiperparâmetros de `train_test_split` em `train_risk_model`.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Treino:", X_train.shape[0], "| Teste:", X_test.shape[0])
print("Positivos — treino:", y_train.mean(), "| teste:", y_test.mean())

## 3. Modelagem preditiva

`make_pipeline()` em `pede_model.py`: `ColumnTransformer` → `RandomForestClassifier` (`n_estimators=400`, `max_depth=14`, `class_weight='balanced_subsample'`, etc.).

In [ ]:
from src.pede_model import make_pipeline

pipe = make_pipeline()
print(pipe)
pipe.fit(X_train, y_train)
print("Treinamento concluído.")

## 4. Avaliação dos resultados

- **ROC-AUC** no conjunto de teste (probabilidade da classe positiva).  
- **Classificação** com limiar 0,5 em `predict_proba`.  
- **Matriz de confusão** para inspeção visual de erros.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC (teste):", roc_auc_score(y_test, proba))
print("\n", classification_report(y_test, pred, digits=3))
print("Matriz de confusão [[TN FP]\n [FN TP]]:")
print(confusion_matrix(y_test, pred))

## 5. (Opcional) Exportar o mesmo artefato do app

`train_risk_model` refaz treino + métricas e calcula defaults para imputação em predições avulsas; `save_bundle` grava `App/models/risk_defasagem.joblib`. Equivale a `python scripts/train_model.py`.

In [ ]:
from src.pede_model import TrainResult, default_model_path, save_bundle, train_risk_model

res: TrainResult = train_risk_model(df)
OUT = default_model_path(ROOT)
save_bundle(OUT, res)
print("Salvo:", OUT)
print("ROC-AUC (holdout do treino completo):", res.metrics.get("roc_auc"))
print(res.metrics["classification_report"])